# MemeX on Kaggle GPU (free T4)

**Setup (once):** upload the `MemeX` project folder as a Kaggle *Dataset* named `memex-code`,
then in this notebook: *Add Input* -> your dataset. Enable GPU T4 x1 in Settings.

Heavy stages (calibration, future-attention labelling, needle benchmark) run 20-50x
faster here than on a 4-core CPU. Results are written to `/kaggle/working` — download
`kv_basis.pt`, `salience_head.pt` and `needle_log.jsonl` back to `D:\MemeX`.

In [ ]:
import subprocess, sys, torch, os, shutil, glob
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers'], check=True)

SRC = glob.glob('/kaggle/input/*/memex') + glob.glob('/kaggle/input/*/*/memex')
assert SRC, 'add the memex-code dataset as an input first'
ROOT = os.path.dirname(SRC[0])
shutil.copytree(ROOT, '/kaggle/working/MemeX', dirs_exist_ok=True)
os.chdir('/kaggle/working/MemeX')
os.makedirs('/kaggle/working/out', exist_ok=True)
print('code at', os.getcwd())

In [ ]:
# donor model + corpora
from huggingface_hub import snapshot_download
MODEL = snapshot_download('Qwen/Qwen3-0.6B', local_dir='/kaggle/working/Qwen3-0.6B')

import urllib.request
os.makedirs('/kaggle/working/data', exist_ok=True)
fic = '/kaggle/working/data/calibration.txt'
if not os.path.exists(fic):
    text = ''
    for u in ['https://www.gutenberg.org/files/2600/2600-0.txt',
              'https://www.gutenberg.org/files/1342/1342-0.txt',
              'https://www.gutenberg.org/files/84/84-0.txt']:
        try:
            text += urllib.request.urlopen(u, timeout=60).read().decode('utf-8', 'ignore')
        except Exception as e:
            print('skip', u, e)
    open(fic, 'w', encoding='utf-8').write(text)
print('fiction chars:', os.path.getsize(fic))

!python memex/make_corpus.py --fiction {fic} --out /kaggle/working/data/corpus_mixed.txt \
    --synth-chars 4000000 --fiction-chars 4000000

In [ ]:
# stage B: spectral transplant basis (minutes on T4 instead of an hour on CPU)
!python memex/transplant.py --model /kaggle/working/Qwen3-0.6B \
    --text /kaggle/working/data/corpus_mixed.txt \
    --out /kaggle/working/out --max-tokens 1000000 --chunk-len 4096 --device cuda

In [ ]:
# future-attention labels + salience head (the flagship research signal)
!python memex/future_attention.py --model /kaggle/working/Qwen3-0.6B \
    --text /kaggle/working/data/corpus_mixed.txt \
    --out /kaggle/working/out/future_attn --max-tokens 600000 --device cuda
!python memex/salience_probe.py --data /kaggle/working/out/future_attn --holdout 4

In [ ]:
# needle benchmark at real scale: long haystack, many cases, all policies
!python bench/needle.py --model /kaggle/working/Qwen3-0.6B \
    --results /kaggle/working/out --futattn /kaggle/working/out/future_attn \
    --haystack 24000 --window 2048 --sinks 4 --notebook 256 \
    --ranks 64 128 256 --depths 0.1 0.35 0.6 0.85 --per-depth 5 \
    --distractors 20 --probe-len 256 --device cuda

In [ ]:
# summary table to copy back into ARCHITECTURE_PLAN.md
import json, collections
tally = collections.defaultdict(list)
for line in open('/kaggle/working/out/needle_log.jsonl'):
    r = json.loads(line)
    tally[r['cond']].append(r['ok'])
for cond, v in sorted(tally.items()):
    print(f'{cond:>16}: {sum(v)}/{len(v)}  ({sum(v)/len(v):.2f})')